In [1]:

import numpy as np
import pandas as pd
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split




## Testing to predict prices :

In [2]:
data_file = Path("/Users/jkzmr/Developer/becode/ImmoEliza/ImmoElizza_team3/data/clean_data_for_model.csv")
df = pd.read_csv(data_file)
df['furnished'] = df['furnished'].astype('bool') # not sure why that got eaten
df = df.drop(columns=['price_by_m2'])


In [3]:
## Setting provinces : 
def set_province():
    conditions = [
        df['zip_code'].between(1000, 1299), # 1 bxl_cap
        df['zip_code'].between(1300, 1499), # 2 brabant_wallon
        df['zip_code'].between(1300, 1499), # 3 brabant_flamand
        df['zip_code'].between(2000, 2999), # 4 anvers
        df['zip_code'].between(3000, 3499), # 3 brabant_flamand #
        df['zip_code'].between(3500, 3999), # 5 limbourg
        df['zip_code'].between(4000, 4999), # 6 liège
        df['zip_code'].between(5000, 5680), # 7 namur
        df['zip_code'].between(6000, 6599), # 8 hainaut
        df['zip_code'].between(6600, 6999), # 9 luxembourg
        df['zip_code'].between(7000, 7999), # 8 hainaut
        df['zip_code'].between(8000, 8999), # 10 flandre_occidentale
        df['zip_code'].between(9000, 9999), # 11 flandre_orientale
    ] # repeating some like hainaut is easier than conditionals
    
    choices = [1, 2, 3, 4, 3, 5, 6, 7, 8, 9, 8, 10, 11]  # one per conditio
    df['province'] = np.select(conditions, choices, default=0)
   
set_province()
df['province'].value_counts()
df = df.drop(columns=['zip_code'])

In [4]:
## One hot Encode of Provinces
df = pd.get_dummies(df, columns=['province'], drop_first=True)
#df = df.drop(columns=['province'])
df.head()


,price,number_of_bedrooms,livable_surface_m2,furnished,has_terrace,has_garden,land_area_m2,number_of_facades,has_swimming_pool,build_year,...,province_2,province_3,province_4,province_5,province_6,province_7,province_8,province_9,province_10,province_11
0,1395000.0,4.0,336.0,True,True,True,5251.0,4.0,False,1955.0,...,False,False,True,False,False,False,False,False,False,False
1,1185000.0,3.0,364.0,False,True,False,NaN,NaN,False,2015.0,...,False,False,True,False,False,False,False,False,False,False
2,572351.0,3.0,138.0,True,False,True,369.0,3.0,False,NaN,...,False,False,True,False,False,False,False,False,False,False
3,1595000.0,5.0,540.0,False,True,True,3896.0,NaN,True,1974.0,...,False,False,True,False,False,False,False,False,False,False
4,549000.0,2.0,149.0,False,True,True,758.0,4.0,False,1986.0,...,False,False,True,False,False,False,False,False,False,False


In [5]:

#df.dropna()

In [6]:
X = df.drop(columns=["price"]) #.to_numpy()  # We need to drop the target column
y = df["price"].to_numpy().reshape(-1 , 1)    # here we do the reshaping in place.
#X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=None, test_size=0.2)


## Checking the shapes:
print("Shape of X: ", X.shape)  
print("Shape of y: ", y.shape)

Shape of X:  (24441, 31)
Shape of y:  (24441, 1)


In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24441 entries, 0 to 24440
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   number_of_bedrooms         23884 non-null  float64
 1   livable_surface_m2         24441 non-null  float64
 2   furnished                  24441 non-null  bool   
 3   has_terrace                24441 non-null  bool   
 4   has_garden                 24441 non-null  bool   
 5   land_area_m2               12555 non-null  float64
 6   number_of_facades          18899 non-null  float64
 7   has_swimming_pool          24441 non-null  bool   
 8   build_year                 14976 non-null  float64
 9   has_garage                 24441 non-null  bool   
 10  number_of_garages          24433 non-null  float64
 11  has_elevator               24441 non-null  bool   
 12  energy_KWh_m2_year         18576 non-null  float64
 13  building_state             19241 non-null  flo

In [8]:

### Filling nulls with median or modes:
from sklearn.base import BaseEstimator, TransformerMixin

class NullFiller(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        ## Getting medians
        self.build_year_median = X['build_year'].median()
        self.energy_median = X['energy_KWh_m2_year'].median()
        self.land_area_median = X['land_area_m2'].median() 
        
        ## Getting modes
        self.build_state_mode = X['building_state'].mode()[0]
        self.num_beds_mode = X['number_of_bedrooms'].mode()[0]
        self.furnished_mode = X['furnished'].mode()[0]
        self.num_facades_mode = X['number_of_facades'].mode()[0]
        self.num_garage_mode = X['number_of_garages'].mode()[0]
    
        return self


    
    def transform(self, X):
        X = X.copy() # this is good practice
        # Writing medians
        X['build_year'] = X['build_year'].fillna(self.build_year_median)
        X['energy_KWh_m2_year'] = X['energy_KWh_m2_year'].fillna(self.energy_median)
        X['land_area_m2'] = X['land_area_m2'].fillna(self.land_area_median)
        
        # Writing modes
        X['building_state'] = X['building_state'].fillna(self.build_state_mode)
        X['number_of_bedrooms'] = X['number_of_bedrooms'].fillna(self.num_beds_mode)
        X['furnished'] = X['furnished'].fillna(self.furnished_mode)
        X['number_of_facades'] = X['number_of_facades'].fillna(self.num_facades_mode)
        X['number_of_garages'] = X['number_of_garages'].fillna(self.num_garage_mode)
        
        ## changing type:
        df['furnished'] = df['furnished'].astype('int')
        

        return X



In [11]:
XGB_model_path = "/Users/jkzmr/Developer/becode/ImmoEliza/ImmoElizza_team3/data/models/XGB_model.pkl"
other_model = "/Users/jkzmr/Developer/becode/ImmoEliza/ImmoElizza_team3/model/model.pkl"

pipe = joblib.load(XGB_model_path)

input_test = {
    "livable_surface_m2": 100,
    "number_of_bedrooms": 2,
    "land_area_m2": None,
    "build_year": 1978,
    "energy_KWh_m2_year": 256.0,
    "has_garden": None,
    "has_terrace": None,
    "has_garage": None,
    "number_of_garages": None,
    "has_elevator": None,
    "has_swimming_pool": None,
    "furnished": None,
    "number_of_facades": None,
    'prop_group_penthouse': False,
    'prop_group_other': False ,
    'prop_group_house' : True,
    'building_state' : False ,
    'prop_group_villa': False ,
    'prop_group_flat': False,
    'prop_group_mixed_building' : False, 
    'province': 1
}

predict_test = pd.DataFrame([input_test])

predictions = pipe.predict(predict_test)
print(predictions)
#print("R² test:", pipe.score(X, y))


[508615.66]
